# Sigorta Hasar Maliyetleri - Keşifsel Veri Analizi (EDA)

Bu notebook, `insurance.csv` veri setini analiz ederek sigorta hasar maliyetlerini (`charges`) etkileyen faktörleri keşfetmeyi amaçlar.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

ModuleNotFoundError: No module named 'pandas'

## 1. Veri Setinin Yuklenmesi ve Genel Bakis

In [ ]:
df = pd.read_csv("insurance.csv")
print(f"Veri seti boyutu: {df.shape[0]} satir, {df.shape[1]} sutun")
df.head(10)

In [ ]:
df.info()

In [ ]:
df.dtypes

## 2. Eksik Veri (Missing Values) Analizi

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Eksik Deger Sayisi": missing,
    "Eksik Deger Orani (%)": missing_pct
})
print(missing_df)
print(f"\nToplam eksik deger: {df.isnull().sum().sum()}")

In [ ]:
# Eksik deger haritasi (gorsel)
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap="viridis", ax=ax)
ax.set_title("Eksik Deger Haritasi")
plt.tight_layout()
plt.show()

## 3. Istatistiksel Ozet

In [ ]:
# Sayisal degiskenler
df.describe()

In [ ]:
# Kategorik degiskenler
df.describe(include="object")

In [ ]:
# Kategorik degiskenlerin dagilimi
for col in df.select_dtypes(include="object").columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

## 4. Hedef Degisken (charges) Dagilimi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df["charges"], bins=40, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_title("Charges Dagilimi")
axes[0].set_xlabel("Charges ($)")
axes[0].set_ylabel("Frekans")

# Box plot
sns.boxplot(y=df["charges"], ax=axes[1], color="steelblue")
axes[1].set_title("Charges Box Plot")

plt.tight_layout()
plt.show()

print(f"Carpiklik (Skewness): {df['charges'].skew():.2f}")
print(f"Basiklik (Kurtosis): {df['charges'].kurtosis():.2f}")

## 5. Sayisal Degiskenlerin Dagilimi

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(2, len(num_cols), figsize=(5 * len(num_cols), 8))

for i, col in enumerate(num_cols):
    axes[0, i].hist(df[col], bins=30, edgecolor="black", alpha=0.7, color="steelblue")
    axes[0, i].set_title(f"{col} - Histogram")
    sns.boxplot(y=df[col], ax=axes[1, i], color="steelblue")
    axes[1, i].set_title(f"{col} - Box Plot")

plt.tight_layout()
plt.show()

## 6. Korelasyon Matrisi

In [ ]:
# Kategorik degiskenleri encode ederek korelasyon hesaplama
df_encoded = df.copy()
df_encoded["sex"] = df_encoded["sex"].map({"male": 1, "female": 0})
df_encoded["smoker"] = df_encoded["smoker"].map({"yes": 1, "no": 0})
df_encoded = pd.get_dummies(df_encoded, columns=["region"], drop_first=True, dtype=int)

corr = df_encoded.corr()

fig, ax = plt.subplots(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Korelasyon Matrisi")
plt.tight_layout()
plt.show()

In [ ]:
# Charges ile korelasyon siralama
print("Charges ile Korelasyon (buyukten kucuge):")
print(corr["charges"].drop("charges").sort_values(ascending=False).to_string())

## 7. Ozellikler ile Charges Arasindaki Iliski

In [ ]:
# Yas - Charges iliskisi (sigara icme durumuna gore)
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker",
                palette={"yes": "red", "no": "steelblue"}, alpha=0.6, ax=ax)
ax.set_title("Yas vs Charges (Sigara Durumuna Gore)")
ax.set_xlabel("Yas")
ax.set_ylabel("Charges ($)")
plt.tight_layout()
plt.show()

In [ ]:
# BMI - Charges iliskisi (sigara icme durumuna gore)
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker",
                palette={"yes": "red", "no": "steelblue"}, alpha=0.6, ax=ax)
ax.set_title("BMI vs Charges (Sigara Durumuna Gore)")
ax.set_xlabel("BMI")
ax.set_ylabel("Charges ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Kategorik degiskenler vs Charges
cat_cols = ["sex", "smoker", "region", "children"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, col in enumerate(cat_cols):
    ax = axes[i // 2, i % 2]
    sns.boxplot(data=df, x=col, y="charges", ax=ax, palette="Set2")
    ax.set_title(f"{col} vs Charges")
    ax.set_ylabel("Charges ($)")

plt.tight_layout()
plt.show()

In [ ]:
# Sigara iceanlerin ve icmeyenlerin ortalama maliyeti
print("Sigara durumuna gore ortalama maliyet:")
print(df.groupby("smoker")["charges"].agg(["mean", "median", "std", "count"]))
print(f"\nSigara icenlerin maliyet orani: "
      f"{df.groupby('smoker')['charges'].mean()['yes'] / df.groupby('smoker')['charges'].mean()['no']:.1f}x")

In [ ]:
# Pair plot (sayisal degiskenler, sigara durumuna gore renklendirilmis)
sns.pairplot(df, hue="smoker", palette={"yes": "red", "no": "steelblue"},
             diag_kind="hist", plot_kws={"alpha": 0.5})
plt.suptitle("Pair Plot (Sigara Durumuna Gore)", y=1.02)
plt.show()

## 8. Ozet Bulgular

Bu bolum notebook calistirildiktan sonra grafiklerden elde edilen bulgularla doldurulabilir.